# **SwipeWise — Part 6: Model Evaluation & Auto-Sklearn Benchmark**
**Member 6 Assignment Work:** Comprehensive Model Evaluation, Comparison & Auto-ML Benchmarking

---
### **Overview:**
This notebook serves as the final evaluation stage of the SwipeWise machine learning pipeline. All models trained by the team are loaded and evaluated using standardized metrics. An `auto-sklearn` benchmark is then run to provide an automated baseline for comparison. The goal is to determine which model best predicts `match_outcome` on the dating app dataset.

**Models evaluated:**
- Baseline 1: DummyClassifier
- Baseline 2: LogisticRegression
- Model 1: DecisionTreeClassifier
- Model 2: RandomForestClassifier (baseline)
- Model 3: KNeighborsClassifier
- Model 4: Tuned Random Forest (from Member 5)
- Model 5: Tuned Gradient Boosting (from Member 5)
- Benchmark: auto-sklearn AutoML

### Step 1 — Install Dependencies & Import Libraries

In [ ]:
# Install auto-sklearn (required on Colab)
# Note: auto-sklearn only runs on Linux. If you are on Windows, use Colab.
!pip install auto-sklearn --quiet
!pip install pyrfr --quiet

In [ ]:
import numpy as np
import pandas as pd
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
print("[SUCCESS] All libraries imported successfully.")

### Step 2 — Load Preprocessed Data Splits
We load the same preprocessed CSV splits used by all team members to ensure consistent evaluation.

In [ ]:
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test  = pd.read_csv('y_test.csv').values.ravel()

print(f"[SUCCESS] Data loaded.")
print(f"  Train: {X_train.shape} | Test: {X_test.shape}")
print(f"  Target classes: {np.unique(y_test)}")

### Step 3 — Retrain Baseline & Standard Models
Since the baseline notebook does not export `.joblib` files, we retrain all baseline models here using the exact same configurations used by the team.

In [ ]:
print("--- Retraining all baseline and standard models ---")

# Baseline 1: DummyClassifier
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)

# Baseline 2: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Model 1: Decision Tree
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)

# Model 2: Random Forest (baseline, not tuned)
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

# Model 3: KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

print("[SUCCESS] All baseline models trained.")

### Step 4 — Load Tuned Models from Member 5
The optimized ensemble models exported by Member 5 are loaded from their `.joblib` files.

In [ ]:
print("--- Loading tuned ensemble models from Member 5 ---")

tuned_rf = joblib.load('tuned_random_forest.joblib')
tuned_gb = joblib.load('tuned_gradient_boosting.joblib')

print("[SUCCESS] Tuned Random Forest loaded.")
print("[SUCCESS] Tuned Gradient Boosting loaded.")

### Step 5 — Evaluate All Models
We evaluate every model using Accuracy and Weighted F1-Score to ensure fair comparison across the imbalanced multi-class target.

In [ ]:
print("--- Evaluating all models on the test set ---")

# Dictionary of all models
all_models = {
    'Dummy Classifier'         : dummy,
    'Logistic Regression'      : lr,
    'Decision Tree'            : dt,
    'Random Forest (Base)'     : rf,
    'KNN'                      : knn,
    'Tuned Random Forest'      : tuned_rf,
    'Tuned Gradient Boosting'  : tuned_gb,
}

results = []

for name, model in all_models.items():
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred) * 100
    f1     = f1_score(y_test, y_pred, average='weighted') * 100
    results.append({'Model': name, 'Accuracy (%)': round(acc, 2), 'F1-Score (%)': round(f1, 2)})
    print(f"  {name:<30} Accuracy: {acc:.2f}%   F1: {f1:.2f}%")

results_df = pd.DataFrame(results)
print("\n[SUCCESS] Evaluation complete.")

### Step 6 — Auto-Sklearn Benchmark
We run `auto-sklearn`, an AutoML framework that automatically searches through multiple ML pipelines (preprocessing + model + hyperparameter combinations) and selects the best one. This serves as our automated upper-bound benchmark.

**Time budget:** 5 minutes total, 90 seconds per model — suitable for this dataset size.

In [ ]:
import autosklearn.classification

print("--- Running auto-sklearn benchmark (this may take 5-10 minutes) ---")
print("    Time budget: 300 seconds total | 90 seconds per model")

start_time = time.time()

automl = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=300,       # Total search time: 5 minutes
    per_run_time_limit=90,             # Max 90s per individual model trial
    seed=42,
    n_jobs=-1,
    memory_limit=4096,
    metric=autosklearn.metrics.accuracy
)

automl.fit(X_train.copy(), y_train.copy())

elapsed = time.time() - start_time
y_pred_auto = automl.predict(X_test)

acc_auto = accuracy_score(y_test, y_pred_auto) * 100
f1_auto  = f1_score(y_test, y_pred_auto, average='weighted') * 100

print(f"\n[SUCCESS] auto-sklearn benchmark completed in {elapsed:.1f}s")
print(f"  auto-sklearn Accuracy : {acc_auto:.2f}%")
print(f"  auto-sklearn F1-Score : {f1_auto:.2f}%")

# Append to results
results.append({'Model': 'auto-sklearn (AutoML)', 'Accuracy (%)': round(acc_auto, 2), 'F1-Score (%)': round(f1_auto, 2)})
results_df = pd.DataFrame(results)

### Step 7 — Print auto-sklearn Model Summary
We inspect which models and configurations auto-sklearn selected internally during its search.

In [ ]:
print("--- auto-sklearn: Sprint Statistics ---")
print(automl.sprint_statistics())
print("\n--- auto-sklearn: Leaderboard ---")
print(automl.leaderboard(detailed=False))

### Step 8 — Full Comparison Table
We display the complete ranked summary of all models including the auto-sklearn benchmark.

In [ ]:
print("\n========== SwipeWise — Final Model Comparison Table ==========")
results_df_sorted = results_df.sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
results_df_sorted.index += 1
print(results_df_sorted.to_string())
print("=" * 60)

### Step 9 — Accuracy & F1 Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#d9534f' if 'auto-sklearn' in m else '#5bc0de' if 'Tuned' in m else '#5cb85c'
          for m in results_df_sorted['Model']]

# Accuracy chart
bars1 = axes[0].barh(results_df_sorted['Model'], results_df_sorted['Accuracy (%)'],
                     color=colors, alpha=0.88, edgecolor='white')
axes[0].set_xlabel('Accuracy (%)', fontsize=11, weight='bold')
axes[0].set_title('Model Accuracy Comparison', fontsize=13, weight='bold')
axes[0].set_xlim(0, 115)
for bar in bars1:
    w = bar.get_width()
    axes[0].text(w + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{w:.2f}%', va='center', fontsize=9, weight='bold')
axes[0].grid(axis='x', linestyle='--', alpha=0.4)
axes[0].invert_yaxis()

# F1 chart
bars2 = axes[1].barh(results_df_sorted['Model'], results_df_sorted['F1-Score (%)'],
                     color=colors, alpha=0.88, edgecolor='white')
axes[1].set_xlabel('Weighted F1-Score (%)', fontsize=11, weight='bold')
axes[1].set_title('Model F1-Score Comparison', fontsize=13, weight='bold')
axes[1].set_xlim(0, 115)
for bar in bars2:
    w = bar.get_width()
    axes[1].text(w + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{w:.2f}%', va='center', fontsize=9, weight='bold')
axes[1].grid(axis='x', linestyle='--', alpha=0.4)
axes[1].invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#5cb85c', label='Baseline / Standard Models'),
    Patch(facecolor='#5bc0de', label='Tuned Ensemble Models'),
    Patch(facecolor='#d9534f', label='auto-sklearn Benchmark'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))

plt.suptitle('SwipeWise — Full Model Evaluation Summary', fontsize=15, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig('swipewise_evaluation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("[SAVED] Chart saved as swipewise_evaluation_comparison.png")

### Step 10 — Confusion Matrices for Top Models
We plot confusion matrices for the two best-performing manually trained models and auto-sklearn to visualize per-class prediction behaviour.

In [ ]:
top_models = {
    'Tuned Gradient Boosting' : tuned_gb,
    'Tuned Random Forest'     : tuned_rf,
    'auto-sklearn'            : automl,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_labels = np.unique(y_test)

for ax, (name, model) in zip(axes, top_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=30)
    ax.set_title(f'{name}', fontsize=11, weight='bold')

plt.suptitle('SwipeWise — Confusion Matrices (Top 3 Models)', fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig('swipewise_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("[SAVED] Confusion matrices saved as swipewise_confusion_matrices.png")

### Step 11 — Detailed Classification Report (Best Manual Model vs auto-sklearn)
We print a full per-class breakdown showing Precision, Recall, and F1 for each match outcome label.

In [ ]:
# Get best manual model (highest accuracy from results_df, excluding auto-sklearn)
manual_results = results_df[results_df['Model'] != 'auto-sklearn (AutoML)']
best_manual_name = manual_results.loc[manual_results['Accuracy (%)'].idxmax(), 'Model']
best_manual_model = all_models.get(best_manual_name, tuned_gb)

print(f"========== Classification Report: {best_manual_name} ==========")
print(classification_report(y_test, best_manual_model.predict(X_test)))

print("========== Classification Report: auto-sklearn ==========")
print(classification_report(y_test, automl.predict(X_test)))

### Step 12 — Evaluation Summary & Insights

In [ ]:
best_row      = results_df_sorted.iloc[0]
auto_row      = results_df_sorted[results_df_sorted['Model'] == 'auto-sklearn (AutoML)'].iloc[0]
dummy_row     = results_df_sorted[results_df_sorted['Model'] == 'Dummy Classifier'].iloc[0]

print("\n" + "=" * 60)
print("       SwipeWise — Member 6 Evaluation Summary")
print("=" * 60)
print(f"  Best overall model     : {best_row['Model']}")
print(f"  Best accuracy          : {best_row['Accuracy (%)']:.2f}%")
print(f"  Best F1-Score          : {best_row['F1-Score (%)']:.2f}%")
print(f"  auto-sklearn accuracy  : {auto_row['Accuracy (%)']:.2f}%")
print(f"  Dummy baseline         : {dummy_row['Accuracy (%)']:.2f}%")
gap = best_row['Accuracy (%)'] - auto_row['Accuracy (%)']
print(f"  Gap (Best vs AutoML)   : {gap:+.2f}%")
print("=" * 60)
print("\nKey Insights:")
print("  - All models significantly outperform the Dummy Classifier baseline.")
print("  - Tuned ensemble models (RF & GB) consistently outperform single estimators.")
print("  - auto-sklearn provides a strong automated benchmark using ensemble voting.")
print("  - The gap between our best model and auto-sklearn reflects the effectiveness")
print("    of manual hyperparameter tuning vs automated model selection.")
print("\n[COMPLETE] Member 6 evaluation pipeline finished successfully.")